In [ ]:
# =============================================================================
# Paper1_11B_MultiMAE_Consensus_13x13
# CELL 1
#
# Clean clustering environment + frozen Fold-3 embedding audit
#
# Kernel:
#     Python (paper1_cluster)
# =============================================================================

import os
import sys

# Keep Windows numerical runtime constrained exactly as before.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn

from sklearn.cluster import KMeans


# -----------------------------------------------------------------------------
# 1. Environment audit
# -----------------------------------------------------------------------------

print("=" * 96)
print("MULTIMAE-STYLE CONSENSUS — ENVIRONMENT AUDIT")
print("=" * 96)

print("Python executable :", sys.executable)
print("Python version    :", sys.version.split()[0])
print("NumPy version     :", np.__version__)
print("SciPy version     :", scipy.__version__)
print("scikit-learn     :", sklearn.__version__)
print("OMP_NUM_THREADS  :", os.environ.get("OMP_NUM_THREADS"))


assert (
    "paper1_cluster"
    in sys.executable
), (
    "Wrong kernel. Select Python (paper1_cluster)."
)


# -----------------------------------------------------------------------------
# 2. Project paths
# -----------------------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\project 3\Paper1_SSL"
)

MULTIMAE_ROOT = (
    PROJECT_ROOT
    / "03_outputs"
    / "modern_baselines"
    / "MultiMAE_13x13"
)

EMBED_DIR = (
    MULTIMAE_ROOT
    / "Fold_3_DEVELOPMENT"
    / "embeddings"
)


TRAIN_EMBED_FILE = (
    EMBED_DIR
    / "MultiMAE_Fold3_TRAIN_13x13_latent64.npy"
)

VAL_EMBED_FILE = (
    EMBED_DIR
    / "MultiMAE_Fold3_VALIDATION_13x13_latent64.npy"
)


assert TRAIN_EMBED_FILE.exists(), TRAIN_EMBED_FILE
assert VAL_EMBED_FILE.exists(), VAL_EMBED_FILE


# -----------------------------------------------------------------------------
# 3. Load frozen raw 64-D embeddings
# -----------------------------------------------------------------------------

Z_train = np.load(
    TRAIN_EMBED_FILE
).astype(
    np.float32
)

Z_val = np.load(
    VAL_EMBED_FILE
).astype(
    np.float32
)


print()
print("=" * 96)
print("FROZEN FOLD-3 EMBEDDINGS")
print("=" * 96)

print("TRAIN shape :", Z_train.shape)
print("VAL shape   :", Z_val.shape)

print(
    "TRAIN finite:",
    np.isfinite(Z_train).all()
)

print(
    "VAL finite  :",
    np.isfinite(Z_val).all()
)


assert Z_train.shape == (
    455,
    64
)

assert Z_val.shape == (
    392,
    64
)

assert np.isfinite(
    Z_train
).all()

assert np.isfinite(
    Z_val
).all()


# -----------------------------------------------------------------------------
# 4. KMeans smoke test using actual MultiMAE embeddings
# -----------------------------------------------------------------------------

km_test = KMeans(
    n_clusters=2,
    random_state=1000,
    n_init=1,
    algorithm="lloyd"
)


test_labels = km_test.fit_predict(
    Z_train[:100]
)


unique_labels, counts = np.unique(
    test_labels,
    return_counts=True
)


print()
print("=" * 96)
print("KMEANS SMOKE TEST")
print("=" * 96)

print(
    "Labels :",
    unique_labels.tolist()
)

print(
    "Counts :",
    counts.tolist()
)

print(
    "Inertia:",
    float(
        km_test.inertia_
    )
)


assert len(
    unique_labels
) == 2

assert counts.sum() == 100

assert np.isfinite(
    km_test.inertia_
)


print()
print("=" * 96)
print(
    "STATUS: MULTIMAE CLUSTERING ENVIRONMENT PASSED"
)
print("=" * 96)

In [ ]:
# =============================================================================
# Paper1_11B_MultiMAE_Consensus_13x13
# CELL 2
#
# Fold-3 DEVELOPMENT consensus clustering for MultiMAE-style baseline
#
# Exact comparison protocol:
#   - TRAIN only for consensus construction
#   - 50 independent KMeans runs per k
#   - n_init = 1
#   - seeds = 1000 ... 1049
#   - k = 2 ... 10
#   - TRAIN co-association matrix
#   - PAC interval: 0.10 < Cij < 0.90
#   - average-linkage consensus partition
#   - consensus centroids in ORIGINAL raw 64-D latent space
#   - VALIDATION assigned afterward to nearest TRAIN consensus centroid
#
# IMPORTANT:
#   k=2...10 is diagnostic only.
#   k=9 remains frozen for direct comparison with PB-MMAE and Vanilla MAE.
# =============================================================================

from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering
)

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)


# -----------------------------------------------------------------------------
# 1. Configuration
# -----------------------------------------------------------------------------

K_VALUES = range(2, 11)

N_RUNS = 50

BASE_SEED = 1000

FROZEN_COMPARISON_K = 9


CONSENSUS_DIR = (
    MULTIMAE_ROOT
    / "Fold_3_DEVELOPMENT"
    / "consensus_k2_k10"
)

CONSENSUS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 100)
print("FOLD 3 — MULTIMAE-STYLE CONSENSUS CLUSTERING")
print("=" * 100)

print("TRAIN latent      :", Z_train.shape)
print("VALIDATION latent :", Z_val.shape)
print("k range           : 2–10")
print("KMeans runs/k     :", N_RUNS)
print("Base seed         :", BASE_SEED)
print("Frozen comparison : k=9")


assert Z_train.shape == (
    455,
    64
)

assert Z_val.shape == (
    392,
    64
)


# -----------------------------------------------------------------------------
# 2. Pairwise initialization stability
# -----------------------------------------------------------------------------

def pairwise_partition_stability(
    label_runs
):

    ari_values = []
    nmi_values = []


    for i, j in combinations(
        range(len(label_runs)),
        2
    ):

        ari_values.append(
            adjusted_rand_score(
                label_runs[i],
                label_runs[j]
            )
        )

        nmi_values.append(
            normalized_mutual_info_score(
                label_runs[i],
                label_runs[j]
            )
        )


    ari_values = np.asarray(
        ari_values,
        dtype=np.float64
    )

    nmi_values = np.asarray(
        nmi_values,
        dtype=np.float64
    )


    return {

        "ARI_mean":
            float(
                np.mean(
                    ari_values
                )
            ),

        "ARI_std":
            float(
                np.std(
                    ari_values,
                    ddof=1
                )
            ),

        "ARI_min":
            float(
                np.min(
                    ari_values
                )
            ),

        "ARI_median":
            float(
                np.median(
                    ari_values
                )
            ),

        "NMI_mean":
            float(
                np.mean(
                    nmi_values
                )
            ),

        "NMI_std":
            float(
                np.std(
                    nmi_values,
                    ddof=1
                )
            ),

        "NMI_min":
            float(
                np.min(
                    nmi_values
                )
            ),

        "NMI_median":
            float(
                np.median(
                    nmi_values
                )
            ),
    }


# -----------------------------------------------------------------------------
# 3. Co-association matrix
# -----------------------------------------------------------------------------

def build_coassociation_matrix(
    label_runs
):

    n_samples = len(
        label_runs[0]
    )


    coassoc = np.zeros(
        (
            n_samples,
            n_samples
        ),
        dtype=np.float32
    )


    for labels in label_runs:

        same_cluster = (
            labels[:, None]
            ==
            labels[None, :]
        )


        coassoc += (
            same_cluster.astype(
                np.float32
            )
        )


    coassoc /= float(
        len(label_runs)
    )


    np.fill_diagonal(
        coassoc,
        1.0
    )


    return coassoc


# -----------------------------------------------------------------------------
# 4. PAC
# -----------------------------------------------------------------------------

def calculate_pac(
    coassoc,
    lower=0.10,
    upper=0.90
):

    values = coassoc[
        np.triu_indices_from(
            coassoc,
            k=1
        )
    ]


    ambiguous = (
        (values > lower)
        &
        (values < upper)
    )


    return float(
        np.mean(
            ambiguous
        )
    )


# -----------------------------------------------------------------------------
# 5. Consensus partition
# -----------------------------------------------------------------------------

def consensus_partition(
    coassoc,
    k
):

    distance = (
        1.0
        -
        coassoc
    )


    np.fill_diagonal(
        distance,
        0.0
    )


    model = AgglomerativeClustering(
        n_clusters=k,
        metric="precomputed",
        linkage="average"
    )


    return model.fit_predict(
        distance
    )


# -----------------------------------------------------------------------------
# 6. Consensus centroids
# -----------------------------------------------------------------------------

def compute_centroids(
    data,
    labels,
    k
):

    centroids = np.zeros(
        (
            k,
            data.shape[1]
        ),
        dtype=np.float64
    )


    for cluster_id in range(
        k
    ):

        subset = data[
            labels
            ==
            cluster_id
        ]


        if len(subset) == 0:

            raise RuntimeError(
                f"Empty TRAIN consensus "
                f"cluster {cluster_id} "
                f"at k={k}"
            )


        centroids[
            cluster_id
        ] = np.mean(
            subset,
            axis=0
        )


    return centroids


# -----------------------------------------------------------------------------
# 7. Held-out validation assignment
# -----------------------------------------------------------------------------

def assign_nearest_centroid(
    data,
    centroids
):

    diff = (
        data[:, None, :]
        -
        centroids[None, :, :]
    )


    squared_distance = np.sum(
        diff * diff,
        axis=2
    )


    return np.argmin(
        squared_distance,
        axis=1
    )


# -----------------------------------------------------------------------------
# 8. Mean within-consensus similarity
# -----------------------------------------------------------------------------

def calculate_mean_within_consensus(
    coassoc,
    labels,
    k
):

    within_values = []


    for cluster_id in range(
        k
    ):

        idx = np.where(
            labels
            ==
            cluster_id
        )[0]


        if len(idx) > 1:

            submatrix = coassoc[
                np.ix_(
                    idx,
                    idx
                )
            ]


            values = submatrix[
                np.triu_indices_from(
                    submatrix,
                    k=1
                )
            ]


            within_values.extend(
                values.tolist()
            )


    if len(within_values) == 0:

        return np.nan


    return float(
        np.mean(
            within_values
        )
    )


# -----------------------------------------------------------------------------
# 9. Run k = 2 ... 10
# -----------------------------------------------------------------------------

records = []

consensus_objects = {}


for k in K_VALUES:

    print()
    print("-" * 100)
    print(f"k = {k}")
    print("-" * 100)


    # -------------------------------------------------------------------------
    # 9A. 50 independent single-init Lloyd KMeans runs
    # -------------------------------------------------------------------------

    label_runs = []


    for run in range(
        N_RUNS
    ):

        km = KMeans(
            n_clusters=k,
            random_state=(
                BASE_SEED
                +
                run
            ),
            n_init=1,
            algorithm="lloyd"
        )


        labels = km.fit_predict(
            Z_train
        )


        label_runs.append(
            labels.astype(
                np.int32
            )
        )


    # -------------------------------------------------------------------------
    # 9B. Pairwise initialization stability
    # -------------------------------------------------------------------------

    stability = (
        pairwise_partition_stability(
            label_runs
        )
    )


    # -------------------------------------------------------------------------
    # 9C. Co-association + PAC
    # -------------------------------------------------------------------------

    coassoc = (
        build_coassociation_matrix(
            label_runs
        )
    )


    pac = calculate_pac(
        coassoc
    )


    # -------------------------------------------------------------------------
    # 9D. Consensus TRAIN partition
    # -------------------------------------------------------------------------

    train_labels = (
        consensus_partition(
            coassoc,
            k
        )
    )


    # -------------------------------------------------------------------------
    # 9E. TRAIN internal cluster metrics
    # -------------------------------------------------------------------------

    silhouette = (
        silhouette_score(
            Z_train,
            train_labels
        )
    )


    dbi = (
        davies_bouldin_score(
            Z_train,
            train_labels
        )
    )


    ch = (
        calinski_harabasz_score(
            Z_train,
            train_labels
        )
    )


    # -------------------------------------------------------------------------
    # 9F. Mean within-consensus similarity
    # -------------------------------------------------------------------------

    mean_within = (
        calculate_mean_within_consensus(
            coassoc,
            train_labels,
            k
        )
    )


    # -------------------------------------------------------------------------
    # 9G. TRAIN cluster counts
    # -------------------------------------------------------------------------

    train_counts = np.bincount(
        train_labels,
        minlength=k
    )


    # -------------------------------------------------------------------------
    # 9H. TRAIN consensus centroids
    # -------------------------------------------------------------------------

    centroids = compute_centroids(
        Z_train,
        train_labels,
        k
    )


    # -------------------------------------------------------------------------
    # 9I. VALIDATION nearest-centroid assignment
    # -------------------------------------------------------------------------

    val_labels = (
        assign_nearest_centroid(
            Z_val,
            centroids
        )
    )


    val_counts = np.bincount(
        val_labels,
        minlength=k
    )


    empty_val_clusters = int(
        np.sum(
            val_counts
            ==
            0
        )
    )


    validation_occupancy = float(
        np.sum(
            val_counts
            >
            0
        )
        /
        k
    )


    validation_dominance = float(
        np.max(
            val_counts
        )
        /
        len(
            val_labels
        )
    )


    # -------------------------------------------------------------------------
    # 9J. Record metrics
    # -------------------------------------------------------------------------

    record = {

        "k":
            k,

        "Runs":
            N_RUNS,

        "ARI_mean":
            stability[
                "ARI_mean"
            ],

        "ARI_std":
            stability[
                "ARI_std"
            ],

        "ARI_min":
            stability[
                "ARI_min"
            ],

        "ARI_median":
            stability[
                "ARI_median"
            ],

        "NMI_mean":
            stability[
                "NMI_mean"
            ],

        "NMI_std":
            stability[
                "NMI_std"
            ],

        "NMI_min":
            stability[
                "NMI_min"
            ],

        "NMI_median":
            stability[
                "NMI_median"
            ],

        "PAC_0.1_0.9":
            pac,

        "Mean_within_consensus":
            mean_within,

        "Silhouette":
            float(
                silhouette
            ),

        "Davies_Bouldin":
            float(
                dbi
            ),

        "Calinski_Harabasz":
            float(
                ch
            ),

        "TRAIN_min_cluster":
            int(
                train_counts.min()
            ),

        "TRAIN_max_cluster":
            int(
                train_counts.max()
            ),

        "VALIDATION_min_cluster":
            int(
                val_counts.min()
            ),

        "VALIDATION_max_cluster":
            int(
                val_counts.max()
            ),

        "Validation_occupancy":
            validation_occupancy,

        "Validation_dominance":
            validation_dominance,

        "Empty_validation_clusters":
            empty_val_clusters,
    }


    records.append(
        record
    )


    # -------------------------------------------------------------------------
    # 9K. Save k-specific outputs
    # -------------------------------------------------------------------------

    k_dir = (
        CONSENSUS_DIR
        / f"k{k}"
    )


    k_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    np.save(
        k_dir
        / (
            f"MultiMAE_Fold3_"
            f"k{k}_coassociation.npy"
        ),
        coassoc
    )


    np.save(
        k_dir
        / (
            f"MultiMAE_Fold3_"
            f"k{k}_TRAIN_labels.npy"
        ),
        train_labels
    )


    np.save(
        k_dir
        / (
            f"MultiMAE_Fold3_"
            f"k{k}_VALIDATION_labels.npy"
        ),
        val_labels
    )


    np.save(
        k_dir
        / (
            f"MultiMAE_Fold3_"
            f"k{k}_consensus_centroids.npy"
        ),
        centroids
    )


    consensus_objects[
        k
    ] = {

        "coassociation":
            coassoc,

        "train_labels":
            train_labels,

        "validation_labels":
            val_labels,

        "centroids":
            centroids,
    }


    # -------------------------------------------------------------------------
    # 9L. Console diagnostics
    # -------------------------------------------------------------------------

    print(
        f"ARI mean/median   = "
        f"{stability['ARI_mean']:.4f} / "
        f"{stability['ARI_median']:.4f}"
    )

    print(
        f"NMI mean/median   = "
        f"{stability['NMI_mean']:.4f} / "
        f"{stability['NMI_median']:.4f}"
    )

    print(
        f"PAC               = "
        f"{pac:.4f}"
    )

    print(
        f"Within consensus  = "
        f"{mean_within:.4f}"
    )

    print(
        f"Silhouette        = "
        f"{silhouette:.4f}"
    )

    print(
        f"DBI               = "
        f"{dbi:.4f}"
    )

    print(
        "TRAIN counts      =",
        train_counts.tolist()
    )

    print(
        "VALIDATION counts =",
        val_counts.tolist()
    )

    print(
        f"Empty VAL clusters= "
        f"{empty_val_clusters}"
    )


# -----------------------------------------------------------------------------
# 10. Summary dataframe
# -----------------------------------------------------------------------------

multimae_fold3_consensus_df = (
    pd.DataFrame(
        records
    )
)


SUMMARY_FILE = (
    CONSENSUS_DIR
    / "MultiMAE_Fold3_k2_k10_consensus_summary.csv"
)


multimae_fold3_consensus_df.to_csv(
    SUMMARY_FILE,
    index=False
)


display_columns = [

    "k",
    "ARI_mean",
    "NMI_mean",
    "PAC_0.1_0.9",
    "Mean_within_consensus",
    "Silhouette",
    "Davies_Bouldin",
    "Validation_occupancy",
    "Empty_validation_clusters",
]


print()
print("=" * 100)
print("FOLD 3 — MULTIMAE-STYLE DEVELOPMENT CONSENSUS SUMMARY")
print("=" * 100)


display(
    multimae_fold3_consensus_df[
        display_columns
    ].round(
        4
    )
)


# -----------------------------------------------------------------------------
# 11. Frozen k=9 development comparison
# -----------------------------------------------------------------------------

multimae_k9_dev = (
    multimae_fold3_consensus_df[
        multimae_fold3_consensus_df[
            "k"
        ]
        ==
        FROZEN_COMPARISON_K
    ]
    .iloc[0]
)


print()
print("=" * 100)
print("FROZEN MULTIMAE-STYLE DEVELOPMENT COMPARISON — k = 9")
print("=" * 100)

print(
    "ARI mean            :",
    f"{multimae_k9_dev['ARI_mean']:.4f}"
)

print(
    "NMI mean            :",
    f"{multimae_k9_dev['NMI_mean']:.4f}"
)

print(
    "PAC                 :",
    f"{multimae_k9_dev['PAC_0.1_0.9']:.4f}"
)

print(
    "Within consensus    :",
    f"{multimae_k9_dev['Mean_within_consensus']:.4f}"
)

print(
    "Silhouette          :",
    f"{multimae_k9_dev['Silhouette']:.4f}"
)

print(
    "DBI                 :",
    f"{multimae_k9_dev['Davies_Bouldin']:.4f}"
)

print(
    "Validation occupancy:",
    f"{multimae_k9_dev['Validation_occupancy']:.4f}"
)

print(
    "Empty VAL clusters  :",
    int(
        multimae_k9_dev[
            "Empty_validation_clusters"
        ]
    )
)


print()
print(
    "Summary saved:"
)

print(
    SUMMARY_FILE
)


# -----------------------------------------------------------------------------
# 12. Hard completion checks
# -----------------------------------------------------------------------------

assert len(
    multimae_fold3_consensus_df
) == 9

assert set(
    multimae_fold3_consensus_df[
        "k"
    ]
) == set(
    range(
        2,
        11
    )
)

assert np.isfinite(
    multimae_fold3_consensus_df[
        [
            "ARI_mean",
            "NMI_mean",
            "PAC_0.1_0.9",
            "Mean_within_consensus",
            "Silhouette",
            "Davies_Bouldin",
        ]
    ].values
).all()


print()
print("=" * 100)
print(
    "STATUS: FOLD-3 MULTIMAE CONSENSUS ANALYSIS PASSED"
)
print("=" * 100)

print(
    "k=2–10 was evaluated diagnostically; "
    "k=9 remains frozen for direct model comparison."
)

In [ ]:
# =============================================================================
# CELL 3
# MultiMAE-style — FIXED k=9 consensus clustering on OUTER FOLDS
#
# Exact comparison protocol:
#   - Outer folds: 1, 2, 4, 5
#   - k = 9 fixed in advance
#   - 50 single-initialization Lloyd KMeans fits on TRAIN only
#   - seeds = 1000 ... 1049
#   - co-association consensus on TRAIN
#   - PAC interval = 0.10–0.90
#   - average-linkage consensus
#   - validation assigned to nearest TRAIN consensus centroid
#
# No outer-fold k search.
# =============================================================================

import os
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering
)

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)


# -----------------------------------------------------------------------------
# 1. Paths and configuration
# -----------------------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\project 3\Paper1_SSL"
)

MULTIMAE_ROOT = (
    PROJECT_ROOT
    / "03_outputs"
    / "modern_baselines"
    / "MultiMAE_13x13"
)

OUTER_FOLDS = [
    1,
    2,
    4,
    5
]

FROZEN_K = 9

N_RUNS = 50

BASE_SEED = 1000


OUTER_CONSENSUS_ROOT = (
    MULTIMAE_ROOT
    / "outer_consensus_k9"
)

OUTER_CONSENSUS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 100)
print("MULTIMAE-STYLE — OUTER-FOLD FIXED-k=9 CONSENSUS")
print("=" * 100)

print("Outer folds :", OUTER_FOLDS)
print("Frozen k    :", FROZEN_K)
print("KMeans runs :", N_RUNS)
print("Base seed   :", BASE_SEED)


# -----------------------------------------------------------------------------
# 2. Helper functions
# -----------------------------------------------------------------------------

def pairwise_partition_stability(
    label_runs
):

    ari_values = []
    nmi_values = []

    for i, j in combinations(
        range(len(label_runs)),
        2
    ):

        ari_values.append(
            adjusted_rand_score(
                label_runs[i],
                label_runs[j]
            )
        )

        nmi_values.append(
            normalized_mutual_info_score(
                label_runs[i],
                label_runs[j]
            )
        )

    ari_values = np.asarray(
        ari_values,
        dtype=np.float64
    )

    nmi_values = np.asarray(
        nmi_values,
        dtype=np.float64
    )

    return {

        "ARI_mean":
            float(
                np.mean(ari_values)
            ),

        "NMI_mean":
            float(
                np.mean(nmi_values)
            ),
    }


def build_coassociation_matrix(
    label_runs
):

    n = len(
        label_runs[0]
    )

    coassoc = np.zeros(
        (n, n),
        dtype=np.float32
    )

    for labels in label_runs:

        same = (
            labels[:, None]
            ==
            labels[None, :]
        )

        coassoc += same.astype(
            np.float32
        )

    coassoc /= float(
        len(label_runs)
    )

    np.fill_diagonal(
        coassoc,
        1.0
    )

    return coassoc


def calculate_pac(
    coassoc,
    lower=0.10,
    upper=0.90
):

    vals = coassoc[
        np.triu_indices_from(
            coassoc,
            k=1
        )
    ]

    ambiguous = (
        (vals > lower)
        &
        (vals < upper)
    )

    return float(
        np.mean(ambiguous)
    )


def consensus_partition(
    coassoc,
    k
):

    distance = (
        1.0
        -
        coassoc
    )

    np.fill_diagonal(
        distance,
        0.0
    )

    model = AgglomerativeClustering(
        n_clusters=k,
        metric="precomputed",
        linkage="average"
    )

    return model.fit_predict(
        distance
    )


def compute_centroids(
    data,
    labels,
    k
):

    centroids = np.zeros(
        (
            k,
            data.shape[1]
        ),
        dtype=np.float64
    )

    for cluster_id in range(
        k
    ):

        subset = data[
            labels
            ==
            cluster_id
        ]

        if len(subset) == 0:

            raise RuntimeError(
                f"Empty TRAIN cluster "
                f"{cluster_id}"
            )

        centroids[
            cluster_id
        ] = np.mean(
            subset,
            axis=0
        )

    return centroids


def assign_nearest_centroid(
    data,
    centroids
):

    diff = (
        data[:, None, :]
        -
        centroids[None, :, :]
    )

    squared_distance = np.sum(
        diff * diff,
        axis=2
    )

    return np.argmin(
        squared_distance,
        axis=1
    )


def mean_within_consensus(
    coassoc,
    labels,
    k
):

    values = []

    for cluster_id in range(
        k
    ):

        idx = np.where(
            labels
            ==
            cluster_id
        )[0]

        if len(idx) > 1:

            sub = coassoc[
                np.ix_(
                    idx,
                    idx
                )
            ]

            vals = sub[
                np.triu_indices_from(
                    sub,
                    k=1
                )
            ]

            values.extend(
                vals.tolist()
            )

    return float(
        np.mean(values)
    )


# -----------------------------------------------------------------------------
# 3. Outer-fold loop
# -----------------------------------------------------------------------------

outer_records = []


for fold in OUTER_FOLDS:

    print()
    print("=" * 100)
    print(f"OUTER FOLD {fold}")
    print("=" * 100)


    # -------------------------------------------------------------------------
    # Load frozen embeddings + metadata
    # -------------------------------------------------------------------------

    embedding_dir = (
        MULTIMAE_ROOT
        / f"Fold_{fold}_OUTER"
        / "embeddings"
    )


    train_embedding_file = (
        embedding_dir
        / (
            f"MultiMAE_Fold{fold}_"
            f"TRAIN_13x13_latent64.npy"
        )
    )


    val_embedding_file = (
        embedding_dir
        / (
            f"MultiMAE_Fold{fold}_"
            f"VALIDATION_13x13_latent64.npy"
        )
    )


    train_meta_file = (
        embedding_dir
        / (
            f"MultiMAE_Fold{fold}_"
            f"TRAIN_13x13_metadata.csv"
        )
    )


    val_meta_file = (
        embedding_dir
        / (
            f"MultiMAE_Fold{fold}_"
            f"VALIDATION_13x13_metadata.csv"
        )
    )


    for path in [
        train_embedding_file,
        val_embedding_file,
        train_meta_file,
        val_meta_file
    ]:

        assert path.exists(), path


    Z_train_fold = np.load(
        train_embedding_file
    ).astype(
        np.float32
    )


    Z_val_fold = np.load(
        val_embedding_file
    ).astype(
        np.float32
    )


    train_meta = pd.read_csv(
        train_meta_file
    )


    val_meta = pd.read_csv(
        val_meta_file
    )


    assert len(
        train_meta
    ) == len(
        Z_train_fold
    )


    assert len(
        val_meta
    ) == len(
        Z_val_fold
    )


    assert np.isfinite(
        Z_train_fold
    ).all()


    assert np.isfinite(
        Z_val_fold
    ).all()


    print(
        "TRAIN latent      :",
        Z_train_fold.shape
    )

    print(
        "VALIDATION latent :",
        Z_val_fold.shape
    )


    # -------------------------------------------------------------------------
    # 50 independent KMeans runs on TRAIN only
    # -------------------------------------------------------------------------

    label_runs = []


    for run in range(
        N_RUNS
    ):

        km = KMeans(
            n_clusters=FROZEN_K,
            random_state=(
                BASE_SEED
                +
                run
            ),
            n_init=1,
            algorithm="lloyd"
        )


        labels = km.fit_predict(
            Z_train_fold
        )


        label_runs.append(
            labels.astype(
                np.int32
            )
        )


    # -------------------------------------------------------------------------
    # Stability
    # -------------------------------------------------------------------------

    stability = (
        pairwise_partition_stability(
            label_runs
        )
    )


    # -------------------------------------------------------------------------
    # Consensus construction
    # -------------------------------------------------------------------------

    coassoc = (
        build_coassociation_matrix(
            label_runs
        )
    )


    pac = calculate_pac(
        coassoc
    )


    train_consensus = (
        consensus_partition(
            coassoc,
            FROZEN_K
        )
    )


    # -------------------------------------------------------------------------
    # TRAIN metrics
    # -------------------------------------------------------------------------

    silhouette = silhouette_score(
        Z_train_fold,
        train_consensus
    )


    dbi = davies_bouldin_score(
        Z_train_fold,
        train_consensus
    )


    ch = calinski_harabasz_score(
        Z_train_fold,
        train_consensus
    )


    within = mean_within_consensus(
        coassoc,
        train_consensus,
        FROZEN_K
    )


    # -------------------------------------------------------------------------
    # TRAIN consensus centroids
    # -------------------------------------------------------------------------

    centroids = compute_centroids(
        Z_train_fold,
        train_consensus,
        FROZEN_K
    )


    # -------------------------------------------------------------------------
    # Held-out VALIDATION assignment
    # -------------------------------------------------------------------------

    val_consensus = (
        assign_nearest_centroid(
            Z_val_fold,
            centroids
        )
    )


    train_counts = np.bincount(
        train_consensus,
        minlength=FROZEN_K
    )


    val_counts = np.bincount(
        val_consensus,
        minlength=FROZEN_K
    )


    empty_val = int(
        np.sum(
            val_counts
            ==
            0
        )
    )


    validation_occupancy = float(
        np.sum(
            val_counts
            >
            0
        )
        /
        FROZEN_K
    )


    val_dominance = float(
        np.max(
            val_counts
        )
        /
        len(
            val_consensus
        )
    )


    # -------------------------------------------------------------------------
    # Save fold outputs
    # -------------------------------------------------------------------------

    fold_out = (
        OUTER_CONSENSUS_ROOT
        / f"Fold_{fold}"
    )


    fold_out.mkdir(
        parents=True,
        exist_ok=True
    )


    np.save(
        fold_out
        / (
            f"MultiMAE_Fold{fold}_"
            f"k9_coassociation.npy"
        ),
        coassoc
    )


    np.save(
        fold_out
        / (
            f"MultiMAE_Fold{fold}_"
            f"k9_TRAIN_labels.npy"
        ),
        train_consensus
    )


    np.save(
        fold_out
        / (
            f"MultiMAE_Fold{fold}_"
            f"k9_VALIDATION_labels.npy"
        ),
        val_consensus
    )


    np.save(
        fold_out
        / (
            f"MultiMAE_Fold{fold}_"
            f"k9_consensus_centroids.npy"
        ),
        centroids
    )


    # -------------------------------------------------------------------------
    # Save Cell_ID labels for later Hungarian alignment
    # -------------------------------------------------------------------------

    required_meta = {
        "Cell_ID",
        "X",
        "Y"
    }


    assert required_meta.issubset(
        train_meta.columns
    ), train_meta.columns


    assert required_meta.issubset(
        val_meta.columns
    ), val_meta.columns


    train_labels_df = (
        train_meta[
            [
                "Cell_ID",
                "X",
                "Y"
            ]
        ].copy()
    )


    train_labels_df[
        "Consensus_k9"
    ] = train_consensus


    train_labels_df[
        "Fold"
    ] = fold


    train_labels_df[
        "Role"
    ] = "TRAIN"


    val_labels_df = (
        val_meta[
            [
                "Cell_ID",
                "X",
                "Y"
            ]
        ].copy()
    )


    val_labels_df[
        "Consensus_k9"
    ] = val_consensus


    val_labels_df[
        "Fold"
    ] = fold


    val_labels_df[
        "Role"
    ] = "VALIDATION"


    train_labels_df.to_csv(
        fold_out
        / (
            f"MultiMAE_Fold{fold}_"
            f"k9_TRAIN_labels_with_CellID.csv"
        ),
        index=False
    )


    val_labels_df.to_csv(
        fold_out
        / (
            f"MultiMAE_Fold{fold}_"
            f"k9_VALIDATION_labels_with_CellID.csv"
        ),
        index=False
    )


    # -------------------------------------------------------------------------
    # Record metrics
    # -------------------------------------------------------------------------

    record = {

        "Fold":
            fold,

        "ARI":
            stability[
                "ARI_mean"
            ],

        "NMI":
            stability[
                "NMI_mean"
            ],

        "PAC":
            pac,

        "Within_consensus":
            within,

        "Silhouette":
            float(
                silhouette
            ),

        "DBI":
            float(
                dbi
            ),

        "Calinski_Harabasz":
            float(
                ch
            ),

        "Validation_occupancy":
            validation_occupancy,

        "Empty_validation_clusters":
            empty_val,

        "Mean_val_dominance":
            val_dominance,

        "TRAIN_min_cluster":
            int(
                train_counts.min()
            ),

        "TRAIN_max_cluster":
            int(
                train_counts.max()
            ),

        "VALIDATION_min_cluster":
            int(
                val_counts.min()
            ),

        "VALIDATION_max_cluster":
            int(
                val_counts.max()
            ),
    }


    outer_records.append(
        record
    )


    print(
        f"ARI              : "
        f"{record['ARI']:.4f}"
    )

    print(
        f"NMI              : "
        f"{record['NMI']:.4f}"
    )

    print(
        f"PAC              : "
        f"{record['PAC']:.4f}"
    )

    print(
        f"Within consensus : "
        f"{record['Within_consensus']:.4f}"
    )

    print(
        f"Silhouette       : "
        f"{record['Silhouette']:.4f}"
    )

    print(
        f"DBI              : "
        f"{record['DBI']:.4f}"
    )

    print(
        f"VAL occupancy    : "
        f"{record['Validation_occupancy']:.4f}"
    )

    print(
        f"Empty VAL        : "
        f"{record['Empty_validation_clusters']}"
    )

    print(
        f"VAL dominance    : "
        f"{record['Mean_val_dominance']:.4f}"
    )

    print(
        "TRAIN counts     :",
        train_counts.tolist()
    )

    print(
        "VALIDATION counts:",
        val_counts.tolist()
    )


# -----------------------------------------------------------------------------
# 4. Per-fold summary
# -----------------------------------------------------------------------------

outer_k9_df = pd.DataFrame(
    outer_records
)


OUTER_SUMMARY_FILE = (
    OUTER_CONSENSUS_ROOT
    / "MultiMAE_outer_folds_fixed_k9_summary.csv"
)


outer_k9_df.to_csv(
    OUTER_SUMMARY_FILE,
    index=False
)


print()
print("=" * 100)
print("MULTIMAE FIXED-k=9 — OUTER-FOLD RESULTS")
print("=" * 100)


display(
    outer_k9_df.round(
        4
    )
)


# -----------------------------------------------------------------------------
# 5. Aggregate
# -----------------------------------------------------------------------------

aggregate = {

    "ARI":
        outer_k9_df[
            "ARI"
        ].mean(),

    "NMI":
        outer_k9_df[
            "NMI"
        ].mean(),

    "PAC":
        outer_k9_df[
            "PAC"
        ].mean(),

    "Within_consensus":
        outer_k9_df[
            "Within_consensus"
        ].mean(),

    "Silhouette":
        outer_k9_df[
            "Silhouette"
        ].mean(),

    "DBI":
        outer_k9_df[
            "DBI"
        ].mean(),

    "Validation_occupancy":
        outer_k9_df[
            "Validation_occupancy"
        ].mean(),

    "Total_empty_validation_clusters":
        int(
            outer_k9_df[
                "Empty_validation_clusters"
            ].sum()
        ),

    "Mean_val_dominance":
        outer_k9_df[
            "Mean_val_dominance"
        ].mean(),
}


aggregate_df = pd.DataFrame(
    [aggregate]
)


AGGREGATE_FILE = (
    OUTER_CONSENSUS_ROOT
    / "MultiMAE_outer_folds_fixed_k9_AGGREGATE.csv"
)


aggregate_df.to_csv(
    AGGREGATE_FILE,
    index=False
)


print()
print("=" * 100)
print("MULTIMAE FIXED-k=9 — OUTER-FOLD AGGREGATE")
print("=" * 100)


display(
    aggregate_df.round(
        4
    )
)


# -----------------------------------------------------------------------------
# 6. Completion checks
# -----------------------------------------------------------------------------

assert set(
    outer_k9_df[
        "Fold"
    ]
) == {
    1,
    2,
    4,
    5
}


assert len(
    outer_k9_df
) == 4


assert np.isfinite(
    outer_k9_df[
        [
            "ARI",
            "NMI",
            "PAC",
            "Within_consensus",
            "Silhouette",
            "DBI"
        ]
    ].values
).all()


print()
print("=" * 100)
print(
    "STATUS: MULTIMAE OUTER-FOLD FIXED-k=9 CONSENSUS PASSED"
)
print("=" * 100)

print()
print(
    "Per-fold summary:"
)

print(
    OUTER_SUMMARY_FILE
)

print()
print(
    "Aggregate summary:"
)

print(
    AGGREGATE_FILE
)

In [ ]:
# =============================================================================
# CELL 4
# MultiMAE k=9 — CROSS-FOLD HUNGARIAN LABEL ALIGNMENT
#
# Exact same-cell alignment protocol used for PB-MMAE and Vanilla MAE:
#   - Fold 1 = reference label system
#   - Folds 2, 4, 5 aligned using identical Cell_ID correspondences
#   - 9 x 9 correspondence matrix
#   - Hungarian assignment maximizes same-cell agreement
#
# IMPORTANT:
#   No latent centroids are compared across independently trained models.
# =============================================================================

from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment


# -----------------------------------------------------------------------------
# 1. Configuration
# -----------------------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\project 3\Paper1_SSL"
)

OUTER_FOLDS = [
    1,
    2,
    4,
    5
]

REFERENCE_FOLD = 1
K = 9


MULTIMAE_ROOT = (
    PROJECT_ROOT
    / "03_outputs"
    / "modern_baselines"
    / "MultiMAE_13x13"
)

CONSENSUS_ROOT = (
    MULTIMAE_ROOT
    / "outer_consensus_k9"
)

ALIGNMENT_ROOT = (
    MULTIMAE_ROOT
    / "crossfold_alignment_k9"
)

ALIGNMENT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 100)
print("MULTIMAE k=9 — CROSS-FOLD HUNGARIAN LABEL ALIGNMENT")
print("=" * 100)

print("Reference fold :", REFERENCE_FOLD)
print("Aligned folds  :", [2, 4, 5])
print("Clusters       :", K)


# -----------------------------------------------------------------------------
# 2. Load complete labeled cell table for each fold
# -----------------------------------------------------------------------------

fold_tables = {}


for fold in OUTER_FOLDS:

    fold_dir = (
        CONSENSUS_ROOT
        / f"Fold_{fold}"
    )

    train_file = (
        fold_dir
        / (
            f"MultiMAE_Fold{fold}_"
            f"k9_TRAIN_labels_with_CellID.csv"
        )
    )

    val_file = (
        fold_dir
        / (
            f"MultiMAE_Fold{fold}_"
            f"k9_VALIDATION_labels_with_CellID.csv"
        )
    )

    assert train_file.exists(), train_file
    assert val_file.exists(), val_file


    train_df = pd.read_csv(
        train_file
    )

    val_df = pd.read_csv(
        val_file
    )


    required = {
        "Cell_ID",
        "Consensus_k9",
        "Fold",
        "Role"
    }


    assert required.issubset(
        train_df.columns
    )

    assert required.issubset(
        val_df.columns
    )


    df = pd.concat(
        [
            train_df,
            val_df
        ],
        ignore_index=True
    )


    df["Cell_ID"] = (
        df["Cell_ID"]
        .astype(str)
        .str.strip()
    )


    # One prediction per geographic cell per model fold
    duplicated = (
        df["Cell_ID"]
        .duplicated()
        .sum()
    )


    assert duplicated == 0, (
        f"Fold {fold}: "
        f"{duplicated} duplicated Cell_ID values"
    )


    df["Consensus_k9"] = (
        df["Consensus_k9"]
        .astype(int)
    )


    labels_present = sorted(
        df["Consensus_k9"]
        .unique()
        .tolist()
    )


    assert labels_present == list(
        range(K)
    ), (
        f"Fold {fold}: unexpected labels "
        f"{labels_present}"
    )


    fold_tables[
        fold
    ] = df


    print(
        f"Fold {fold}: "
        f"{len(df)} labeled cells"
    )


# -----------------------------------------------------------------------------
# 3. Fold 1 reference
# -----------------------------------------------------------------------------

reference = (
    fold_tables[
        REFERENCE_FOLD
    ][
        [
            "Cell_ID",
            "Consensus_k9"
        ]
    ]
    .rename(
        columns={
            "Consensus_k9":
                "Reference_label"
        }
    )
)


reference_mapping = {
    i: i
    for i in range(K)
}


# -----------------------------------------------------------------------------
# 4. Hungarian alignment
# -----------------------------------------------------------------------------

mapping_records = []

alignment_diagnostics = []


for fold in OUTER_FOLDS:

    current = (
        fold_tables[
            fold
        ].copy()
    )


    if fold == REFERENCE_FOLD:

        mapping = (
            reference_mapping
        )

        current[
            "Aligned_k9"
        ] = current[
            "Consensus_k9"
        ]

        shared_n = len(
            current
        )

        agreement_before = 1.0
        agreement_after = 1.0


    else:

        shared = reference.merge(
            current[
                [
                    "Cell_ID",
                    "Consensus_k9"
                ]
            ],
            on="Cell_ID",
            how="inner"
        )


        shared_n = len(
            shared
        )


        assert shared_n > 0


        correspondence = np.zeros(
            (
                K,
                K
            ),
            dtype=np.int64
        )


        for ref_label, cur_label in zip(
            shared[
                "Reference_label"
            ],
            shared[
                "Consensus_k9"
            ]
        ):

            correspondence[
                int(ref_label),
                int(cur_label)
            ] += 1


        row_ind, col_ind = (
            linear_sum_assignment(
                -correspondence
            )
        )


        # Current-fold label -> Fold-1 label
        mapping = {

            int(current_label):
                int(reference_label)

            for reference_label,
                current_label
            in zip(
                row_ind,
                col_ind
            )
        }


        assert set(
            mapping.keys()
        ) == set(
            range(K)
        )

        assert set(
            mapping.values()
        ) == set(
            range(K)
        )


        current[
            "Aligned_k9"
        ] = (
            current[
                "Consensus_k9"
            ]
            .map(
                mapping
            )
            .astype(int)
        )


        agreement_before = float(
            np.mean(
                shared[
                    "Reference_label"
                ].to_numpy()
                ==
                shared[
                    "Consensus_k9"
                ].to_numpy()
            )
        )


        shared_aligned = (
            reference.merge(
                current[
                    [
                        "Cell_ID",
                        "Aligned_k9"
                    ]
                ],
                on="Cell_ID",
                how="inner"
            )
        )


        agreement_after = float(
            np.mean(
                shared_aligned[
                    "Reference_label"
                ].to_numpy()
                ==
                shared_aligned[
                    "Aligned_k9"
                ].to_numpy()
            )
        )


        correspondence_df = (
            pd.DataFrame(
                correspondence,

                index=[
                    f"Fold1_{i}"
                    for i in range(K)
                ],

                columns=[
                    f"Fold{fold}_{i}"
                    for i in range(K)
                ]
            )
        )


        correspondence_df.to_csv(
            ALIGNMENT_ROOT
            / (
                f"MultiMAE_Fold1_vs_Fold{fold}_"
                f"k9_correspondence_matrix.csv"
            )
        )


    # -------------------------------------------------------------------------
    # Store aligned table
    # -------------------------------------------------------------------------

    fold_tables[
        fold
    ] = current


    # -------------------------------------------------------------------------
    # Record mapping
    # -------------------------------------------------------------------------

    for original_label in range(
        K
    ):

        mapping_records.append(
            {
                "Fold":
                    fold,

                "Original_k9":
                    original_label,

                "Aligned_k9":
                    mapping[
                        original_label
                    ]
            }
        )


    alignment_diagnostics.append(
        {
            "Fold":
                fold,

            "Shared_cells_with_Fold1":
                shared_n,

            "Agreement_before_alignment":
                agreement_before,

            "Agreement_after_alignment":
                agreement_after
        }
    )


    print()
    print(
        f"Fold {fold}"
    )

    print(
        "Shared cells with Fold 1 :",
        shared_n
    )

    print(
        "Mapping                 :",
        mapping
    )

    print(
        "Agreement before        :",
        f"{agreement_before:.4f}"
    )

    print(
        "Agreement after         :",
        f"{agreement_after:.4f}"
    )


# -----------------------------------------------------------------------------
# 5. Save alignment mappings + diagnostics
# -----------------------------------------------------------------------------

mapping_df = pd.DataFrame(
    mapping_records
)

diagnostic_df = pd.DataFrame(
    alignment_diagnostics
)


MAPPING_FILE = (
    ALIGNMENT_ROOT
    / "MultiMAE_k9_Hungarian_label_mappings.csv"
)

DIAGNOSTIC_FILE = (
    ALIGNMENT_ROOT
    / "MultiMAE_k9_Hungarian_alignment_diagnostics.csv"
)


mapping_df.to_csv(
    MAPPING_FILE,
    index=False
)

diagnostic_df.to_csv(
    DIAGNOSTIC_FILE,
    index=False
)


# -----------------------------------------------------------------------------
# 6. Build aligned long-format table
# -----------------------------------------------------------------------------

aligned_long_parts = []


for fold in OUTER_FOLDS:

    df = (
        fold_tables[
            fold
        ].copy()
    )


    keep_columns = [
        "Cell_ID",
        "Fold",
        "Role",
        "Consensus_k9",
        "Aligned_k9"
    ]


    for col in [
        "X",
        "Y"
    ]:

        if col in df.columns:

            keep_columns.append(
                col
            )


    aligned_long_parts.append(
        df[
            keep_columns
        ]
    )


aligned_long = pd.concat(
    aligned_long_parts,
    ignore_index=True
)


ALIGNED_LONG_FILE = (
    ALIGNMENT_ROOT
    / "MultiMAE_k9_aligned_labels_long.csv"
)


aligned_long.to_csv(
    ALIGNED_LONG_FILE,
    index=False
)


# -----------------------------------------------------------------------------
# 7. Audit
# -----------------------------------------------------------------------------

print()
print("=" * 100)
print("ALIGNMENT DIAGNOSTICS")
print("=" * 100)


display(
    diagnostic_df.round(
        4
    )
)


print()
print("=" * 100)
print("LABEL PERMUTATIONS")
print("=" * 100)


display(
    mapping_df
)


print()
print(
    "Aligned rows:",
    len(
        aligned_long
    )
)

print(
    "Unique Cell_IDs:",
    aligned_long[
        "Cell_ID"
    ].nunique()
)


print()
print(
    "Aligned long file:"
)

print(
    ALIGNED_LONG_FILE
)


# -----------------------------------------------------------------------------
# 8. Hard checks
# -----------------------------------------------------------------------------

assert set(
    aligned_long[
        "Fold"
    ]
) == {
    1,
    2,
    4,
    5
}


assert aligned_long[
    "Aligned_k9"
].between(
    0,
    8
).all()


assert not aligned_long[
    [
        "Cell_ID",
        "Fold",
        "Aligned_k9"
    ]
].isna().any().any()


# We expect the same total spatial support used by PB-MMAE and Vanilla MAE.
assert (
    aligned_long[
        "Cell_ID"
    ].nunique()
    ==
    1477
)


print()
print("=" * 100)
print(
    "STATUS: MULTIMAE k=9 CROSS-FOLD HUNGARIAN ALIGNMENT PASSED"
)
print("=" * 100)

print(
    "Independent latent centroids were NOT compared across folds."
)

In [ ]:
# =============================================================================
# CELL 5
# MultiMAE-style — EXACT PB-MMAE substitution hierarchy
#
# Identical mathematics to PB-MMAE and Vanilla MAE:
#
#   1. Same-cell cross-model prediction pairs
#   2. Symmetric-support confusion counts
#   3. Row-normalized directional confusion
#   4. S_ij = 0.5 * (P_ij + P_ji)
#   5. D_ij = 1 - S_ij
#   6. Average-linkage hierarchical clustering
#   7. Flat cuts m = 2...8 using criterion="maxclust"
#   8. Cell-wise and pairwise cross-fold reproducibility
#
# NO cross-model latent-centroid distances are used.
# =============================================================================

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from scipy.cluster.hierarchy import (
    linkage,
    fcluster
)

from scipy.spatial.distance import squareform

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score
)


# -----------------------------------------------------------------------------
# 1. Configuration
# -----------------------------------------------------------------------------

K_FINE = 9

OUTER_FOLDS = [
    1,
    2,
    4,
    5
]


PROJECT_ROOT = Path(
    r"C:\project 3\Paper1_SSL"
)

MULTIMAE_ROOT = (
    PROJECT_ROOT
    / "03_outputs"
    / "modern_baselines"
    / "MultiMAE_13x13"
)

ALIGNMENT_ROOT = (
    MULTIMAE_ROOT
    / "crossfold_alignment_k9"
)

INPUT_FILE = (
    ALIGNMENT_ROOT
    / "MultiMAE_k9_aligned_labels_long.csv"
)

OUTPUT_DIR = (
    ALIGNMENT_ROOT
    / "metadomain_analysis"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


assert INPUT_FILE.exists(), INPUT_FILE


print("=" * 100)
print("MULTIMAE k=9 CROSS-FOLD SUBSTITUTION / META-DOMAIN ANALYSIS")
print("=" * 100)

print("Input :", INPUT_FILE)
print("Output:", OUTPUT_DIR)


# -----------------------------------------------------------------------------
# 2. Load aligned-label table
# -----------------------------------------------------------------------------

raw_df = pd.read_csv(
    INPUT_FILE
)


required_columns = {
    "Cell_ID",
    "Fold",
    "Aligned_k9"
}


assert required_columns.issubset(
    raw_df.columns
), raw_df.columns


df = raw_df.copy()


df["Cell_ID"] = (
    df["Cell_ID"]
    .astype(str)
    .str.strip()
)


df = df.rename(
    columns={
        "Fold":
            "Model_Fold",

        "Aligned_k9":
            "Aligned_cluster_k9"
    }
)


df["Model_Fold"] = (
    df["Model_Fold"]
    .astype(int)
)


df["Aligned_cluster_k9"] = (
    df["Aligned_cluster_k9"]
    .astype(int)
)


assert df[
    "Aligned_cluster_k9"
].between(
    0,
    K_FINE - 1
).all()


assert set(
    df["Model_Fold"].unique()
) == set(
    OUTER_FOLDS
)


print()
print(
    "Rows           :",
    len(df)
)

print(
    "Unique Cell_IDs:",
    df["Cell_ID"].nunique()
)

print(
    "Model folds    :",
    sorted(
        df["Model_Fold"]
        .unique()
        .tolist()
    )
)


# -----------------------------------------------------------------------------
# 3. Build cross-model pairs on identical geographic cells
# -----------------------------------------------------------------------------

pair_records = []


for cell_id, group in df.groupby(
    "Cell_ID"
):

    if len(group) < 2:
        continue


    rows = list(
        group.itertuples(
            index=False
        )
    )


    for a, b in combinations(
        rows,
        2
    ):

        label_a = int(
            a.Aligned_cluster_k9
        )

        label_b = int(
            b.Aligned_cluster_k9
        )


        pair_records.append(
            {
                "Cell_ID":
                    cell_id,

                "Model_A":
                    int(
                        a.Model_Fold
                    ),

                "Model_B":
                    int(
                        b.Model_Fold
                    ),

                "Label_A":
                    label_a,

                "Label_B":
                    label_b,

                "Agreement":
                    int(
                        label_a
                        ==
                        label_b
                    )
            }
        )


pair_df = pd.DataFrame(
    pair_records
)


assert len(pair_df) > 0


PAIR_FILE = (
    OUTPUT_DIR
    / "MultiMAE_k9_shared_cell_model_pairs.csv"
)


pair_df.to_csv(
    PAIR_FILE,
    index=False
)


print()
print(
    "Cross-model same-cell pairs:",
    len(pair_df)
)

print(
    "Raw fine-label pairwise agreement:",
    round(
        pair_df[
            "Agreement"
        ].mean(),
        4
    )
)


# -----------------------------------------------------------------------------
# 4. Symmetric-support confusion counts
# -----------------------------------------------------------------------------

confusion = np.zeros(
    (
        K_FINE,
        K_FINE
    ),
    dtype=np.float64
)


for row in pair_df.itertuples(
    index=False
):

    i = int(
        row.Label_A
    )

    j = int(
        row.Label_B
    )


    confusion[
        i,
        j
    ] += 1.0


    confusion[
        j,
        i
    ] += 1.0


confusion_df = pd.DataFrame(
    confusion,

    index=[
        f"C{i}"
        for i in range(
            K_FINE
        )
    ],

    columns=[
        f"C{i}"
        for i in range(
            K_FINE
        )
    ]
)


CONFUSION_FILE = (
    OUTPUT_DIR
    / "MultiMAE_k9_crossfold_confusion_counts.csv"
)


confusion_df.to_csv(
    CONFUSION_FILE
)


# -----------------------------------------------------------------------------
# 5. Exact normalized substitution matrix
#
# P_ij = confusion_ij / sum_j(confusion_ij)
#
# S_ij = 0.5 * (P_ij + P_ji)
# -----------------------------------------------------------------------------

row_sums = confusion.sum(
    axis=1,
    keepdims=True
)


with np.errstate(
    divide="ignore",
    invalid="ignore"
):

    directional = np.divide(

        confusion,

        row_sums,

        out=np.zeros_like(
            confusion
        ),

        where=(
            row_sums != 0
        )
    )


substitution = (
    0.5
    *
    (
        directional
        +
        directional.T
    )
)


np.fill_diagonal(
    substitution,
    1.0
)


substitution_df = pd.DataFrame(
    substitution,

    index=[
        f"C{i}"
        for i in range(
            K_FINE
        )
    ],

    columns=[
        f"C{i}"
        for i in range(
            K_FINE
        )
    ]
)


SUBSTITUTION_FILE = (
    OUTPUT_DIR
    / "MultiMAE_k9_crossfold_substitution_similarity.csv"
)


substitution_df.to_csv(
    SUBSTITUTION_FILE
)


# -----------------------------------------------------------------------------
# 6. Edge list
# -----------------------------------------------------------------------------

edge_records = []


for i in range(
    K_FINE
):

    for j in range(
        i + 1,
        K_FINE
    ):

        edge_records.append(
            {
                "Cluster_A":
                    i,

                "Cluster_B":
                    j,

                "Substitution_similarity":
                    float(
                        substitution[
                            i,
                            j
                        ]
                    ),

                "Raw_confusion_count":
                    float(
                        confusion[
                            i,
                            j
                        ]
                        +
                        confusion[
                            j,
                            i
                        ]
                    )
            }
        )


edge_df = pd.DataFrame(
    edge_records
)


edge_df = (
    edge_df
    .sort_values(
        "Substitution_similarity",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


EDGE_FILE = (
    OUTPUT_DIR
    / "MultiMAE_k9_cluster_confusion_graph_edges.csv"
)


edge_df.to_csv(
    EDGE_FILE,
    index=False
)


# -----------------------------------------------------------------------------
# 7. Hierarchical clustering
#
# D = 1 - S
# average linkage
# -----------------------------------------------------------------------------

distance = (
    1.0
    -
    substitution
)


distance = np.clip(
    distance,
    0.0,
    1.0
)


np.fill_diagonal(
    distance,
    0.0
)


condensed_distance = squareform(
    distance,
    checks=False
)


Z = linkage(
    condensed_distance,
    method="average"
)


linkage_df = pd.DataFrame(
    Z,
    columns=[
        "Cluster_1",
        "Cluster_2",
        "Distance",
        "New_cluster_size"
    ]
)


LINKAGE_FILE = (
    OUTPUT_DIR
    / "MultiMAE_k9_metadomain_hierarchical_linkage.csv"
)


linkage_df.to_csv(
    LINKAGE_FILE,
    index=False
)


# -----------------------------------------------------------------------------
# 8. Evaluation function
# -----------------------------------------------------------------------------

def evaluate_mapping(
    fine_to_meta,
    n_meta
):

    temp = df[
        [
            "Cell_ID",
            "Model_Fold",
            "Aligned_cluster_k9"
        ]
    ].copy()


    temp[
        "Meta_domain"
    ] = (
        temp[
            "Aligned_cluster_k9"
        ]
        .map(
            fine_to_meta
        )
        .astype(int)
    )


    # -------------------------------------------------------------------------
    # Cell-wise majority agreement
    # -------------------------------------------------------------------------

    cell_records = []


    for cell_id, group in temp.groupby(
        "Cell_ID"
    ):

        labels = group[
            "Meta_domain"
        ].to_numpy()


        counts = np.bincount(
            labels,
            minlength=n_meta
        )


        max_count = int(
            counts.max()
        )


        winners = np.where(
            counts
            ==
            max_count
        )[0]


        n_models = len(
            labels
        )


        tied = (
            len(winners)
            >
            1
        )


        if tied:

            consensus = -1

        else:

            consensus = int(
                winners[0]
            )


        cell_records.append(
            {
                "Cell_ID":
                    cell_id,

                "N_models":
                    n_models,

                "Consensus_meta_domain":
                    consensus,

                "Tie":
                    tied,

                "Agreement_fraction":
                    float(
                        max_count
                        /
                        n_models
                    ),

                "Full_agreement":
                    bool(
                        max_count
                        ==
                        n_models
                    )
            }
        )


    cell_df = pd.DataFrame(
        cell_records
    )


    # -------------------------------------------------------------------------
    # Pairwise model agreement
    # -------------------------------------------------------------------------

    pair_agreements = []
    pair_aris = []
    pair_nmis = []


    for fold_a, fold_b in combinations(
        OUTER_FOLDS,
        2
    ):

        A = (
            temp[
                temp[
                    "Model_Fold"
                ]
                ==
                fold_a
            ][
                [
                    "Cell_ID",
                    "Meta_domain"
                ]
            ]
            .rename(
                columns={
                    "Meta_domain":
                        "Meta_domain_A"
                }
            )
        )


        B = (
            temp[
                temp[
                    "Model_Fold"
                ]
                ==
                fold_b
            ][
                [
                    "Cell_ID",
                    "Meta_domain"
                ]
            ]
            .rename(
                columns={
                    "Meta_domain":
                        "Meta_domain_B"
                }
            )
        )


        shared = A.merge(
            B,
            on="Cell_ID",
            how="inner"
        )


        assert len(
            shared
        ) > 0


        y_a = shared[
            "Meta_domain_A"
        ].to_numpy()


        y_b = shared[
            "Meta_domain_B"
        ].to_numpy()


        pair_agreements.append(
            np.mean(
                y_a
                ==
                y_b
            )
        )


        pair_aris.append(
            adjusted_rand_score(
                y_a,
                y_b
            )
        )


        pair_nmis.append(
            normalized_mutual_info_score(
                y_a,
                y_b
            )
        )


    return {

        "N_meta_domains":
            n_meta,

        "Mean_cell_agreement":
            float(
                cell_df[
                    "Agreement_fraction"
                ].mean()
            ),

        "Median_cell_agreement":
            float(
                cell_df[
                    "Agreement_fraction"
                ].median()
            ),

        "Full_agreement_fraction":
            float(
                cell_df[
                    "Full_agreement"
                ].mean()
            ),

        "Tie_fraction":
            float(
                cell_df[
                    "Tie"
                ].mean()
            ),

        "Mean_pairwise_label_accuracy":
            float(
                np.mean(
                    pair_agreements
                )
            ),

        "Mean_pairwise_ARI":
            float(
                np.mean(
                    pair_aris
                )
            ),

        "Mean_pairwise_NMI":
            float(
                np.mean(
                    pair_nmis
                )
            ),

        "Cell_table":
            cell_df
    }


# -----------------------------------------------------------------------------
# 9. Fine k=9 cross-fold reproducibility
# -----------------------------------------------------------------------------

fine_identity_mapping = {
    i: i
    for i in range(
        K_FINE
    )
}


fine_k9_result = evaluate_mapping(
    fine_identity_mapping,
    K_FINE
)


print()
print("=" * 100)
print("FINE k=9 CROSS-FOLD REPRODUCIBILITY")
print("=" * 100)

print(
    "Mean cell agreement     :",
    f"{fine_k9_result['Mean_cell_agreement']:.4f}"
)

print(
    "Median cell agreement   :",
    f"{fine_k9_result['Median_cell_agreement']:.4f}"
)

print(
    "Full agreement fraction :",
    f"{fine_k9_result['Full_agreement_fraction']:.4f}"
)

print(
    "Tie fraction            :",
    f"{fine_k9_result['Tie_fraction']:.4f}"
)

print(
    "Mean pairwise accuracy  :",
    f"{fine_k9_result['Mean_pairwise_label_accuracy']:.4f}"
)

print(
    "Mean pairwise ARI       :",
    f"{fine_k9_result['Mean_pairwise_ARI']:.4f}"
)

print(
    "Mean pairwise NMI       :",
    f"{fine_k9_result['Mean_pairwise_NMI']:.4f}"
)


# -----------------------------------------------------------------------------
# 10. Flat cuts m = 2 ... 8
# -----------------------------------------------------------------------------

candidate_records = []

candidate_mappings = {}

candidate_cell_tables = {}


for n_meta in range(
    2,
    K_FINE
):

    cluster_membership = fcluster(

        Z,

        t=n_meta,

        criterion="maxclust"
    )


    # scipy 1-based -> 0-based
    cluster_membership = (
        cluster_membership
        -
        1
    )


    fine_to_meta = {

        fine_cluster:
            int(
                cluster_membership[
                    fine_cluster
                ]
            )

        for fine_cluster in range(
            K_FINE
        )
    }


    candidate_mappings[
        n_meta
    ] = fine_to_meta


    result = evaluate_mapping(
        fine_to_meta,
        n_meta
    )


    candidate_cell_tables[
        n_meta
    ] = result[
        "Cell_table"
    ]


    candidate_records.append(
        {
            "N_meta_domains":
                n_meta,

            "Mean_cell_agreement":
                result[
                    "Mean_cell_agreement"
                ],

            "Median_cell_agreement":
                result[
                    "Median_cell_agreement"
                ],

            "Full_agreement_fraction":
                result[
                    "Full_agreement_fraction"
                ],

            "Tie_fraction":
                result[
                    "Tie_fraction"
                ],

            "Mean_pairwise_label_accuracy":
                result[
                    "Mean_pairwise_label_accuracy"
                ],

            "Mean_pairwise_ARI":
                result[
                    "Mean_pairwise_ARI"
                ],

            "Mean_pairwise_NMI":
                result[
                    "Mean_pairwise_NMI"
                ],

            "Agreement_gain_vs_k9":
                (
                    result[
                        "Mean_cell_agreement"
                    ]
                    -
                    fine_k9_result[
                        "Mean_cell_agreement"
                    ]
                ),

            "Full_agreement_gain_vs_k9":
                (
                    result[
                        "Full_agreement_fraction"
                    ]
                    -
                    fine_k9_result[
                        "Full_agreement_fraction"
                    ]
                )
        }
    )


candidate_df = pd.DataFrame(
    candidate_records
)


# -----------------------------------------------------------------------------
# 11. Save fine -> meta mappings
# -----------------------------------------------------------------------------

mapping_records = []


for n_meta, mapping in (
    candidate_mappings.items()
):

    for fine_cluster in range(
        K_FINE
    ):

        mapping_records.append(
            {
                "N_meta_domains":
                    n_meta,

                "Fine_cluster_k9":
                    fine_cluster,

                "Meta_domain":
                    mapping[
                        fine_cluster
                    ]
            }
        )


mapping_df = pd.DataFrame(
    mapping_records
)


MAPPING_FILE = (
    OUTPUT_DIR
    / "MultiMAE_k9_fine_to_metadomain_mappings.csv"
)


mapping_df.to_csv(
    MAPPING_FILE,
    index=False
)


# -----------------------------------------------------------------------------
# 12. Save candidate stability table
# -----------------------------------------------------------------------------

CANDIDATE_FILE = (
    OUTPUT_DIR
    / "MultiMAE_k9_metadomain_candidate_stability.csv"
)


candidate_df.to_csv(
    CANDIDATE_FILE,
    index=False
)


# -----------------------------------------------------------------------------
# 13. Save fixed m=4 cell-wise result
# -----------------------------------------------------------------------------

M4_CELL_FILE = (
    OUTPUT_DIR
    / "MultiMAE_k9_m4_cellwise_consensus.csv"
)


candidate_cell_tables[
    4
].to_csv(
    M4_CELL_FILE,
    index=False
)


# -----------------------------------------------------------------------------
# 14. Strongest substitutions
# -----------------------------------------------------------------------------

print()
print("=" * 100)
print("STRONGEST MULTIMAE FINE-CLUSTER SUBSTITUTIONS")
print("=" * 100)


display(
    edge_df[
        [
            "Cluster_A",
            "Cluster_B",
            "Substitution_similarity",
            "Raw_confusion_count"
        ]
    ]
    .head(
        15
    )
    .round(
        4
    )
)


# -----------------------------------------------------------------------------
# 15. Linkage table
# -----------------------------------------------------------------------------

print()
print("=" * 100)
print("MULTIMAE HIERARCHICAL LINKAGE")
print("=" * 100)


display(
    linkage_df.round(
        6
    )
)


# -----------------------------------------------------------------------------
# 16. m = 2 ... 8 stability
# -----------------------------------------------------------------------------

print()
print("=" * 100)
print("MULTIMAE META-DOMAIN STABILITY: m = 2...8")
print("=" * 100)


display(
    candidate_df.round(
        {
            "Mean_cell_agreement":
                4,

            "Median_cell_agreement":
                4,

            "Full_agreement_fraction":
                4,

            "Tie_fraction":
                4,

            "Mean_pairwise_label_accuracy":
                4,

            "Mean_pairwise_ARI":
                4,

            "Mean_pairwise_NMI":
                4,

            "Agreement_gain_vs_k9":
                4,

            "Full_agreement_gain_vs_k9":
                4
        }
    )
)


# -----------------------------------------------------------------------------
# 17. Print fine -> meta mappings
# -----------------------------------------------------------------------------

print()
print("=" * 100)
print("FINE k=9 -> MULTIMAE META-DOMAIN MAPPINGS")
print("=" * 100)


for n_meta in range(
    2,
    K_FINE
):

    mapping = candidate_mappings[
        n_meta
    ]


    groups = {}


    for fine_cluster, meta_domain in (
        mapping.items()
    ):

        groups.setdefault(
            meta_domain,
            []
        ).append(
            fine_cluster
        )


    print(
        f"\n{n_meta} meta-domains:"
    )


    for meta_domain in sorted(
        groups
    ):

        print(
            f"  M{meta_domain}: "
            f"{groups[meta_domain]}"
        )


# -----------------------------------------------------------------------------
# 18. Fixed m=4 summary
# -----------------------------------------------------------------------------

m4 = (
    candidate_df[
        candidate_df[
            "N_meta_domains"
        ]
        ==
        4
    ]
    .iloc[0]
)


print()
print("=" * 100)
print("FIXED MULTIMAE FOUR-DOMAIN RESULT — m = 4")
print("=" * 100)

print(
    "Mean cell agreement     :",
    f"{m4['Mean_cell_agreement']:.4f}"
)

print(
    "Median cell agreement   :",
    f"{m4['Median_cell_agreement']:.4f}"
)

print(
    "Full agreement fraction :",
    f"{m4['Full_agreement_fraction']:.4f}"
)

print(
    "Tie fraction            :",
    f"{m4['Tie_fraction']:.4f}"
)

print(
    "Mean pairwise accuracy  :",
    f"{m4['Mean_pairwise_label_accuracy']:.4f}"
)

print(
    "Mean pairwise ARI       :",
    f"{m4['Mean_pairwise_ARI']:.4f}"
)

print(
    "Mean pairwise NMI       :",
    f"{m4['Mean_pairwise_NMI']:.4f}"
)


print()
print(
    "m=4 fine->meta mapping:",
    candidate_mappings[
        4
    ]
)


# -----------------------------------------------------------------------------
# 19. m=4 -> m=5 reproducibility change
# -----------------------------------------------------------------------------

m5 = (
    candidate_df[
        candidate_df[
            "N_meta_domains"
        ]
        ==
        5
    ]
    .iloc[0]
)


print()
print("=" * 100)
print("MULTIMAE m=4 -> m=5 REPRODUCIBILITY CHANGE")
print("=" * 100)

print(
    "Mean agreement change :",
    f"{m5['Mean_cell_agreement'] - m4['Mean_cell_agreement']:+.4f}"
)

print(
    "Full agreement change :",
    f"{m5['Full_agreement_fraction'] - m4['Full_agreement_fraction']:+.4f}"
)

print(
    "Tie fraction change   :",
    f"{m5['Tie_fraction'] - m4['Tie_fraction']:+.4f}"
)


# -----------------------------------------------------------------------------
# 20. Hard checks
# -----------------------------------------------------------------------------

assert len(
    candidate_df
) == 7


assert set(
    candidate_df[
        "N_meta_domains"
    ]
) == set(
    range(
        2,
        9
    )
)


assert substitution.shape == (
    9,
    9
)


assert np.allclose(
    substitution,
    substitution.T
)


assert np.allclose(
    np.diag(
        substitution
    ),
    1.0
)


assert Z.shape == (
    8,
    4
)


assert len(
    candidate_cell_tables[
        4
    ]
) == 1477


print()
print("=" * 100)
print(
    "STATUS: MULTIMAE EXACT CROSS-FOLD META-DOMAIN ANALYSIS PASSED"
)
print("=" * 100)

print()
print(
    "No cross-model latent-centroid distances were used."
)

print(
    "m=2...8 are flat cuts of one substitution-derived "
    "average-linkage dendrogram."
)

print()
print(
    "Candidate table:",
    CANDIDATE_FILE
)

print(
    "m=4 cell table:",
    M4_CELL_FILE
)

print(
    "Linkage table:",
    LINKAGE_FILE
)

print(
    "Substitution matrix:",
    SUBSTITUTION_FILE
)

In [ ]:
# =============================================================================
# CELL 6
# Freeze final contemporary-baseline comparison
# =============================================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    r"C:\project 3\Paper1_SSL"
)

OUT_FILE = (
    PROJECT_ROOT
    / "03_outputs"
    / "modern_baselines"
    / "FINAL_modern_baseline_comparison.csv"
)

comparison_df = pd.DataFrame(
    [
        {
            "Model": "PB-MMAE",
            "Outer_k9_ARI": 0.4857,
            "Outer_k9_NMI": 0.6515,
            "Outer_k9_PAC": 0.2324,
            "Outer_k9_Within": 0.6065,
            "Outer_k9_Silhouette": 0.1446,
            "Outer_k9_DBI": 2.1060,
            "Validation_occupancy": 0.6111,
            "Empty_validation_clusters": 14,
            "Validation_dominance": 0.4656,
            "M4_mean_cell_agreement": 0.8230,
            "M4_full_agreement": 0.5170,
            "M4_tie_fraction": 0.0640,
            "M4_pairwise_accuracy": 0.6270,
            "M4_to_M5_mean_change": -0.0600,
            "M4_to_M5_full_change": -0.1020,
            "M4_to_M5_tie_change": 0.0900,
        },
        {
            "Model": "Vanilla MAE",
            "Outer_k9_ARI": 0.7348,
            "Outer_k9_NMI": 0.8174,
            "Outer_k9_PAC": 0.1228,
            "Outer_k9_Within": 0.7895,
            "Outer_k9_Silhouette": 0.3412,
            "Outer_k9_DBI": 0.9209,
            "Validation_occupancy": 0.4722,
            "Empty_validation_clusters": 19,
            "Validation_dominance": 0.5401,
            "M4_mean_cell_agreement": 0.8037,
            "M4_full_agreement": 0.5369,
            "M4_tie_fraction": 0.1544,
            "M4_pairwise_accuracy": 0.6240,
            "M4_to_M5_mean_change": -0.0260,
            "M4_to_M5_full_change": -0.0691,
            "M4_to_M5_tie_change": 0.0088,
        },
        {
            "Model": "MultiMAE-style",
            "Outer_k9_ARI": 0.6903,
            "Outer_k9_NMI": 0.7850,
            "Outer_k9_PAC": 0.1410,
            "Outer_k9_Within": 0.7698,
            "Outer_k9_Silhouette": 0.3036,
            "Outer_k9_DBI": 1.0469,
            "Validation_occupancy": 0.5556,
            "Empty_validation_clusters": 16,
            "Validation_dominance": 0.5589,
            "M4_mean_cell_agreement": 0.8243,
            "M4_full_agreement": 0.5328,
            "M4_tie_fraction": 0.1029,
            "M4_pairwise_accuracy": 0.6627,
            "M4_to_M5_mean_change": -0.0338,
            "M4_to_M5_full_change": -0.0555,
            "M4_to_M5_tie_change": 0.0535,
        },
    ]
)

comparison_df.to_csv(
    OUT_FILE,
    index=False
)

display(
    comparison_df.round(4)
)

print()
print("Saved:")
print(OUT_FILE)

print()
print(
    "STATUS: FINAL MODERN-BASELINE COMPARISON FROZEN"
)